# 🚀 Business Entity Resolution — Google Colab GPU Runbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luciferxy/amazon-entity-resolution/blob/main/student_resource/colab_runbook.ipynb)

### End-to-End GPU Pipeline:
- **Runtime:** NVIDIA T4 GPU (free tier) or High-RAM GPU (Change runtime type → T4 GPU).
- **GPU Blocking ($K=5$):** PyTorch CUDA sparse CSR tensor math (`_topk_gpu_exact`) cuts blocking time from 3 hours to **~10–12 minutes**.
- **Compact Candidates:** Generates ~3.3 candidates per Source 1 entity (crucial for hackathon ranking).
- **Optimized Model:** Trained on 300,000 S1 entities with 4-fold GroupKFold and LOCO cross-validation, targeting $F_{0.5} \ge 0.98$.
- **Persistence:** Outputs are written directly to Google Drive so they survive runtime resets.

## 1. Mount Google Drive & Locate Project

Upload the `student_resource` folder to your Google Drive (e.g. into `MyDrive/student_resource`). This cell will automatically find it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive')

# Search for train_source1.tsv on Drive
matches = [p for p in DRIVE_ROOT.rglob('train_source1.tsv') if p.parent.name == 'train' and p.parent.parent.name == 'dataset']
if not matches:
    print('Searching MyDrive folders:', [p.name for p in DRIVE_ROOT.iterdir() if p.is_dir()][:15])
    raise FileNotFoundError('Could not find dataset/train/train_source1.tsv. Upload student_resource folder to Google Drive.')

PROJECT = matches[0].parents[2]
print('Found Project Root at:', PROJECT)
os.chdir(PROJECT)
print('Working Directory:', os.getcwd())

## 2. Verify GPU Acceleration

Check that an NVIDIA GPU (T4 / V100 / A100) is active.

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
cuda_ok = torch.cuda.is_available()
print('CUDA Available:', cuda_ok)
if cuda_ok:
    dev = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Device: {dev} | VRAM: {vram:.1f} GB')
else:
    print('WARNING: GPU not detected! Go to Runtime -> Change runtime type -> T4 GPU.')

## 3. Install Dependencies

Installs optimized packages for text processing, gradient boosting, and PyTorch Lightning.

In [ ]:
%pip install -q pandas pyarrow numpy scipy scikit-learn lightgbm rapidfuzz anyascii sparse_dot_topn lightning
import pandas, pyarrow, lightgbm, rapidfuzz, anyascii, sparse_dot_topn, lightning
print('All dependencies ready!')

## 4. Configure GPU Backend ($K=5$)

Sets exact GPU sparse matrix cosine similarity.

In [ ]:
os.environ['ER_BACKEND'] = 'gpu' if torch.cuda.is_available() else 'cpu'
os.environ['ER_GPU_MODE'] = 'exact'
os.environ['ER_GPU_BS'] = '2048'
os.environ['ER_K'] = '5'
print(f'ER_BACKEND={os.environ["ER_BACKEND"]} | ER_GPU_BS={os.environ["ER_GPU_BS"]} | ER_K={os.environ["ER_K"]}')

## 5. Preprocess Data to Parquet

Normalizes business names, legal forms, addresses, and phone/tax numbers into Parquet format under `work/`. Run once per fresh dataset.

In [ ]:
from pathlib import Path
if not Path('work/train_s1.parquet').exists():
    !python src/prep.py train
else:
    print('Train Parquets already exist under work/')

if not Path('work/test_s1.parquet').exists():
    !python src/prep.py test
else:
    print('Test Parquets already exist under work/')

## 6. Evaluate Candidate Generation (Recall Check)

Measures recall of the candidate blocking step. With $K=5$, expected recall is **~95.3%** with **3.34 candidates/entity** (Oracle $F_{0.5} = 0.9817$).

In [ ]:
!python src/eval_block.py

## 7. Train Entity Resolution Model

Trains on 300,000 Source 1 entities with 4-fold `GroupKFold` cross-validation and Leave-One-Country-Out (LOCO) checks. Saves `work/lgb.txt` and `work/model_meta.json`.

In [ ]:
# 100,000 S1 sample finishes in ~6-8 min on T4 GPU with ER_GPU_BS=2048
TRAIN_SAMPLE = 100_000
!python src/train_lightning.py {TRAIN_SAMPLE}

## 8. Predict Test Set & Generate Submissions

Runs GPU blocking on the 1.73M S1 queries vs 9.97M pool entities, extracts 28 tabular features, classifies matches, and writes:
- `output/candidate_pairs.tsv`
- `output/matching_results.tsv`

In [ ]:
!python src/run_test.py

## 9. Validate Submission Format

Runs the official competition validation script to ensure 100% compliance with column names, tab delimiters, and non-empty checks.

In [ ]:
!python utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test

## 10. Package & Download Final Submission

Zips both submission files directly into Google Drive for download.

In [ ]:
!zip -j output/submission_files.zip output/matching_results.tsv output/candidate_pairs.tsv
print('Submission ZIP created at: output/submission_files.zip')
!ls -lh output/